# Instationäre Behälterleerung -- die Gleichung programmieren

**BIW3-10 Weiterführende Hydromechanik · Übung 2 · Aufgabe 5**

In der Übung haben wir die Behälterleerung analytisch gelöst (Trennung der Variablen, Folie 26):

$$A\,\frac{\mathrm{d}h}{\mathrm{d}t} = -\alpha A_{\mathrm{o}}\sqrt{2gh} \qquad\Longrightarrow\qquad h(t) = h_0\left(1-\frac{t}{T_E}\right)^2,\qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

**Hier programmiert ihr die Gleichung selbst** und lasst sie von `scipy` in der Zeit rechnen. Derspringende Punkt der Instationarität steckt in der rechten Seite: Sie bewertet Torricelli immer mit dem **momentanen** Wasserstand $h(t)$ -- jede Auswertung ist eine Momentaufnahme (quasi-stationär, Folie 26).

**Lernziele:** die DGL als Funktion `f(t, y)` aufschreiben · mit `scipy.integrate.solve_ivp` lösen · $h(t)$ und $Q(t)$ als Zeitreihen lesen · erklären, was die Rechnung instationär macht.

## 1. Parameter (wie in Aufgabe 4 des Übungsblatts)

Zylinderspeicher mit scharfkantiger Bodenöffnung, kein Zulauf. Alle Werte in **SI-Einheiten** (Meter, Sekunden).

In [ ]:
# ======================================================================
# 1. PARAMETER DER AUFGABE 4 (Übungsblatt) -- alle in SI-Einheiten!
# ======================================================================
# Jede Zeile: Name = Wert   # Einheit | Bedeutung

g     = 9.81    # m/s^2  | Erdbeschleunigung -- treibt die Ausflussgeschwindigkeit
D     = 2.0     # m      | Behälter-Innendurchmesser (Zylinder)
d_o   = 0.08    # m      | Durchmesser der Bodenöffnung (Blatt: 80 mm -> in Meter)
alpha = 0.62    # -      | Ausflusszahl (Folie 16): Strahlverengung + Verluste
h0    = 3.6     # m      | Anfangswasserstand über der Öffnung -- unsere Unbekannte h(t)

In [ ]:
# ======================================================================
# 2. ABGELEITETE GRÖSSEN - werden aus den Parametern berechnet
# ======================================================================
import numpy as np                 # Numerik-Bibliothek (sqrt, pi, Arrays, ...)
import matplotlib.pyplot as plt    # Plot-Bibliothek

# --- Flächen (Kreisfläche A = pi * d^2 / 4) ---------------------------
A   = np.pi * D**2 / 4    # m^2 | Behälterquerschnitt
A_o = np.pi * d_o**2 / 4  # m^2 | Öffnungsquerschnitt

# --- kappa: Koeffizient VOR der Wurzel in der DGL ---------------------
#   Volumenbilanz + Torricelli (Aufgabe 4b):  A * dh/dt = -alpha*A_o*sqrt(2 g h)
#   Durch A teilen:                           dh/dt = -kappa * sqrt(h)
kappa = alpha * A_o * np.sqrt(2*g) / A    # sqrt(m)/s

# --- Analytische Auslaufzeit (Übungsblatt, Aufgabe 4c/d) --------------
T_E = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))    # s

# --- Anfangsabfluss: Torricelli mit dem ANFÄNGLICHEN Wasserstand ------
Q0 = alpha * A_o * np.sqrt(2*g*h0)    # m^3/s

# --- Kontrolle: sollten die Werte aus dem Übungsblatt sein ------------
print(f"A       = {A:.4f} m^2        (Blatt: 3,142 m^2)")
print(f"A_o     = {A_o*1e3:.3f} e-3 m^2 (Blatt: 5,03 e-3 m^2)")
print(f"kappa   = {kappa:.3e} sqrt(m)/s")
print(f"T_E     = {T_E:.1f} s = {T_E/60:.2f} min   (Blatt: 863,6 s = 14,4 min)")
print(f"Q_0     = {Q0*1000:.1f} L/s       (Blatt: 26,2 L/s)")

## 2. Die Gleichung programmieren (Aufgabe 5a)

Die **rechte Seite** der DGL enthält die ganze Physik -- sie sagt, wie schnell sich $h$ *momentan* ändert:

$$\frac{\mathrm{d}h}{\mathrm{d}t} = f(t,h) = -\underbrace{\frac{\alpha A_{\mathrm{o}}\sqrt{2gh}}{A}}_{Q_{\mathrm{ab}}(t)/A} = -\kappa\,\sqrt{h}, \qquad \kappa = \frac{\alpha A_{\mathrm{o}}\sqrt{2g}}{A} \approx 4{,}39\cdot 10^{-3}\ \sqrt{\mathrm{m}}/\mathrm{s}$$

- **Torricelli** gibt den Abfluss zum *momentanen* Spiegel: $Q_{\mathrm{ab}} = \alpha A_{\mathrm{o}}\sqrt{2gh}$ -- deshalb ist $f$ von $h$ abhängig,
- **Kontinuität** teilt durch $A$: ein Volumenstrom $Q$ senkt den Spiegel um $Q/A$ pro Zeit,
- $Q_{\mathrm{zu}} = 0$ (nichts läuft nach).

`scipy` wird später diese Funktion an vielen Zeitpunkten auswerten -- wir liefern nur das "Physik-Gesetz".

In [ ]:
# ======================================================================
# 3. DIE GLEICHUNG PROGRAMMIEREN   (Aufgabe 5a -- LÖSUNG)
# ======================================================================
def rhs(t, y):
    """Rechte Seite der Behälter-DGL für solve_ivp.
        t : Zeit [s] (unbenutzt),  y : Zustandsvektor, y[0] = h [m].
        Rückgabe: dh/dt [m/s].
    """
    h = y[0]

    # TODO 1 (gelöst): Torricelli mit dem momentanen Wasserstand
    Q_ab = alpha * A_o * np.sqrt(2 * g * max(h, 0.0))     # m^3/s

    # TODO 2 (gelöst): Kontinuität -- Q_ab senkt den Spiegel um Q_ab/A
    dhdt = -Q_ab / A                                      # m/s

    return dhdt

# Mini-Test: Anfangssteigung (soll -8,3 mm/s sein, siehe Musterlösung 4a)
print(f"dh/dt(0) = {rhs(0, [h0])*1000:.2f} mm/s")

## 3. Mit scipy lösen (Aufgabe 5b)

`scipy.integrate.solve_ivp` integriert $\dd y/\dd t = f(t,y)$ automatisch (Runge-Kutta 4/5, **adaptive Schrittweite** -- groß, wo sich wenig tut; klein, wo es steil wird). Wir übergeben:

- die **rechte Seite** `rhs` (eure Funktion von oben),
- den **Zeitbereich** `(0, 3*T_E)` -- sicherheitshalber bis zum Dreifachen,
- den **Anfangszustand** `[h0]` (Vektor! später wären mehrere Unbekannte möglich),
- ein **Ereignis (Event)**: "Behälter leer" bei $h = 0$ -- dort stoppt die Rechnung,
- Genauigkeit: `rtol=1e-8`, `atol=1e-10`.

In [ ]:
# ======================================================================
# 4. MIT SCIPY LÖSEN   (Aufgabe 5b -- LÖSUNG)
# ======================================================================
from scipy.integrate import solve_ivp

def event_leer(t, y):
    return y[0]                     # TODO 3 (gelöst): Event bei h = 0

event_leer.terminal = True          # bei h = 0 anhalten
event_leer.direction = -1           # nur + -> - Übergänge zählen

t_span = (0, 3*T_E)                 # Sicherheitszugabe
y0 = [h0]                           # Anfangsbedingung als VEKTOR
solver = dict(rtol=1e-8, atol=1e-10)

# TODO 4 (gelöst): Zeitintegration
sol = solve_ivp(rhs, t_span, y0, events=event_leer, **solver)

# TODO 5 (gelöst): Auslaufzeit aus dem Event
T_E_num = sol.t_events[0][0]

print(f"T_E,num   = {T_E_num:.2f} s")
print(f"T_E,exakt = {T_E:.2f} s")
print(f"Abweichung = {(T_E_num - T_E)/T_E*100:+.4f} %")

# Zusatz: Anfangs- und Endverhalten der Zeitableitung (Instationarität!)
print(f"dh/dt(0)      = {rhs(0, [h0])*1000:+.2f} mm/s  (maximale Absenkung)")
print(f"dh/dt(0.5T_E) = {rhs(T_E/2, [h0/4])*1000:+.2f} mm/s  (halber Abfluss)")

## 4. Instationarität sichtbar machen (Aufgabe 5c)

Die gelöste Zeitreihe $h(t)$ ist die Instationarität zum Anfassen: am festen Ort (dem Behälter) ändert sich der Zustand ständig -- $\partial h/\partial t \neq 0$. Auch der Abfluss $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$ fällt mit $h$.

In [ ]:
# ======================================================================
# 5. ANALYTISCHE LÖSUNG ZUM VERGLEICH (aus der Übung, Aufgabe 4c)
# ======================================================================
def h_analytisch(t):
    """Exakter Spiegelverlauf h(t) [m]: Parabel aus der Trennung der Variablen."""
    return h0 * (1 - t/T_E)**2

def Q_analytisch(t):
    """Abfluss Q(t) [m^3/s]: Torricelli mit dem momentanen h (linearer Abfall)."""
    return alpha * A_o * np.sqrt(2*g*np.maximum(h_analytisch(t), 0.0))

# Stützpunkte der numerischen Lösung (solve_ivp liefert sol.t und sol.y[0]):
t_num = sol.t
h_num = sol.y[0]

In [ ]:
# ======================================================================
# 6. PLOT   (Aufgabe 5c -- LÖSUNG)
# ======================================================================
plt.figure(figsize=(8, 4))
plt.plot(t_num, h_num, 'b-', linewidth=2, label='numerisch: h(t)')

# TODO 6 (gelöst): analytische Lösung überlagern
t_fein = np.linspace(0, T_E, 200)
plt.plot(t_fein, h_analytisch(t_fein), 'r--', linewidth=1.5, label='exakt (Aufgabe 4c)')

# TODO 7 (gelöst): Sensor einzeichnen
plt.axhline(1.0, color='orange', linestyle='--', label='Sensor (1,0 m)')

plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Instationäre Behälterleerung')
plt.show()

# Ablesbar: Sensor wird bei t ~ 408 s (6,8 min) erreicht -- Aufgabe 4e.
# Nach der halben Zeit ist h = h0/4 = 0,90 m (Kontrolle durch Hinsehen).

## 5. Interpretation (Aufgabe 5d) – Erwartete Antworten

1. **Instationär ist die Rechnung, weil $\partial h/\partial t \neq 0$:** am festen Ort (Behälter) ändert sich der Zustand mit der Zeit -- $h(t)$ fällt, $Q(t)$ fällt von 26,2 L/s auf 0. Der Zeitableitungs-Term $A\,\dd h/\dd t$ ist der neue Term gegenüber der stationären Bilanz.
2. **$\dd h/\dd t = -\kappa\sqrt{h}$ ist am Anfang am größten** ($h = h_0$ am größten $\Rightarrow$ $Q$ am größten, 8,3 mm/s Absenkung), am Ende fast null ($h\to 0 \Rightarrow Q \to 0$). Deshalb die Parabel: steil -- flach.
3. **Pumpe (Aufgabe 3a):** `rhs` würde zu $-Q_{\mathrm{p}}/A = -k$ (konstant) $\Rightarrow$ $h(t) = h_0 - kt$ -- **linearer** Spiegelfall. Die Form von $h(t)$ steckt komplett in der rechten Seite.
4. **Quasi-stationär:** `rhs` bewertet Torricelli mit dem momentanen $h$ -- jedes Zeitschritt-Intervall ist eine Momentaufnahme (Folie 26). Zulässig, weil sich der Spiegel langsam gegen die Anpassungszeit der Strömung ändert ($v_{\mathrm{S}} \ll v_{\mathrm{o}}$, Aufgabe 4a).
5. **Kontrolle:** $h(T_{\mathrm{E}}/2) = h_0/4 = 0{,}90$ m -- im Plot ablesbar; exakt der Wert aus der analytischen Rechnung.